#Reading From Bronze Table

# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DataType, DoubleType, IntegerType
from pyspark.sql.functions import trim, col
from pyspark.sql.functions import split, col
from pyspark.sql.window import Window

In [0]:
RENAME_MAP_CUST = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

In [0]:
df = spark.table('dev_project.bronze.crm_cust_info')

# Data Transformations

## Customer Information Table

###Trimming Excess White Spaces

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Normalization

In [0]:
df =  (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(df.cst_marital_status) == "S", "Single")
        .when(F.upper(df.cst_marital_status) == "M", "Married")
        .when(F.upper(df.cst_marital_status) == "D", "Divorced")
        .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(df.cst_gndr) == "M", "Male")
        .when(F.upper(df.cst_gndr) == "F", "Female")
        .otherwise("n/a")
    )
)

### Renaming the Columns

In [0]:
for old_col, new_col in RENAME_MAP_CUST.items():
    df = df.withColumnRenamed(old_col, new_col)

### Removing Records With Missing Cust ID

In [0]:
df = df.filter(col("customer_id").isNotNull())

In [0]:
# Dedup on customer_key alone — source contains current state only
# Order by _file_name descending as no ingestion timestamp exists on this table
window_dedup = Window.partitionBy("customer_key") \
                     .orderBy(F.col("_file_name").desc())

df = df.withColumn("_row_num", F.row_number().over(window_dedup)) \
       .filter(F.col("_row_num") == 1) \
       .drop("_row_num")

In [0]:
# SHA-256 surrogate key on (customer_key + current_date)
# Deterministic per version — each new version on a new date gets a unique key
df = df.withColumn(
    "customer_surrogate_key",
    F.sha2(
        F.concat_ws("||",
            F.col("customer_key"),
            F.current_date().cast("string")
        ),
        256
    )
)

# SCD2 date columns — managed by us, not from source
# start_date and end_date will be set properly during the merge below
# We define them here so the schema is consistent for the write
df = df.withColumn("start_date", F.current_date()) \
       .withColumn("end_date", F.lit(None).cast("date")) \
       .withColumn("is_current", F.lit(True)) \
       .withColumn("customer_version", F.lit(1))

In [0]:
# Explicit final select — drops Bronze metadata columns
df_customers_silver = df.select(
    "customer_surrogate_key",
    "customer_id",
    "customer_key",
    "first_name",
    "last_name",
    "marital_status",
    "gender",
    "created_date",
    "start_date",
    "end_date",
    "is_current",
    "customer_version"
)

In [0]:
print("\n" + "="*60)
print(" CUSTOMER DATA QUALITY VALIDATION")
print("="*60)
print(f"Total records:        {df_customers_silver.count():,}")
print(f"Current versions:     {df_customers_silver.filter(F.col('is_current') == True).count():,}")
print(f"Historical versions:  {df_customers_silver.filter(F.col('is_current') == False).count():,}")

dup_check = df_customers_silver.groupBy("customer_key").count().filter("count > 1")
print(f"Duplicate keys:       {dup_check.count()}")
if dup_check.count() > 0:
    dup_check.show()

null_keys = df_customers_silver.filter(F.col("customer_key").isNull()).count()
print(f"Null customer keys:   {null_keys}")

null_ids = df_customers_silver.filter(F.col("customer_id").isNull()).count()
print(f"Null customer IDs:    {null_ids}")
print("="*60 + "\n")

# Write Customer Table into Silver Layer

In [0]:
from delta.tables import DeltaTable

TARGET_TABLE = "dev_project.silver.customer_info"
today = F.current_date()

# --- FIRST RUN: Table doesn't exist yet — write and exit ---
if not spark.catalog.tableExists(TARGET_TABLE):
    df_customers_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(TARGET_TABLE)
    print("Initial load complete — table created.")

else:
    silver = DeltaTable.forName(spark, TARGET_TABLE)

    # Step 1: Hash business columns on incoming batch
    df_incoming = df_customers_silver.withColumn(
        "incoming_hash",
        F.sha2(F.concat_ws("||",
            F.col("first_name"),
            F.col("last_name"),
            F.col("marital_status"),
            F.col("gender")
        ), 256)
    )

    # Step 2: Hash business columns on existing Silver current records only
    df_existing = silver.toDF().withColumn(
        "existing_hash",
        F.sha2(F.concat_ws("||",
            F.col("first_name"),
            F.col("last_name"),
            F.col("marital_status"),
            F.col("gender")
        ), 256)
    ).filter(F.col("is_current") == True) \
     .select("customer_key", "existing_hash")

    # Step 3: Classify incoming records
    df_classified = df_incoming.join(df_existing, on="customer_key", how="left")

    df_new     = df_classified.filter(F.col("existing_hash").isNull())
    df_changed = df_classified.filter(
                     F.col("existing_hash").isNotNull() &
                     (F.col("incoming_hash") != F.col("existing_hash"))
                 )

    # Step 4: Close changed records in Silver
    if not df_changed.isEmpty():
        changed_keys = [r["customer_key"] for r in df_changed.select("customer_key").collect()]

        silver.update(
            condition=(
                (F.col("is_current") == True) &
                (F.col("customer_key").isin(changed_keys))
            ),
            set={
                "is_current": F.lit(False),
                "end_date":   today
            }
        )

    # Step 5: Insert new + changed records with correct version numbers
    df_to_insert = df_new.unionByName(df_changed)

    if not df_to_insert.isEmpty():
        # Get max existing version per customer_key from Silver
        df_max_version = silver.toDF() \
            .groupBy("customer_key") \
            .agg(F.max("customer_version").alias("max_version"))

        df_to_insert = df_to_insert \
            .join(df_max_version, on="customer_key", how="left") \
            .withColumn(
                "customer_version",
                F.coalesce(F.col("max_version"), F.lit(0)) + F.lit(1)
            ) \
            .drop("max_version", "incoming_hash", "existing_hash")

        df_to_insert.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(TARGET_TABLE)

    print(f"SCD2 write complete — New: {df_new.count()}, Changed: {df_changed.count()}")

In [0]:
%sql
SELECT * FROM dev_project.silver.customer_info LIMIT 100